**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**03 | Participant-Grouped Unimodal Movement Classification**

![Source-aligned sEMG and acceleration, classical baselines, CNN/TCN/BiGRU encoders and grouped Development evaluation](../../images/03_unimodal_grouped_baselines.svg)

**Research question:** How do sEMG-only and acceleration-only classical and temporal classifiers compare when evaluated on the same previously unseen Development participants?

**Inputs:** The fixed NinaPro DB2 E1 **40,000 paired windows** from Notebook 02, with **35 Development participants (35,000 windows)** and **five reserved Test participants**. Each observation has one of **18 original corrected source labels (0–17)**.

**Study design:** Five participant-disjoint Development folds evaluate a majority-prior baseline, two Logistic Regression models, two Random Forest models and six 1D-CNN/TCN/BiGRU unimodal architectures. Feature scaling and neural early stopping use permitted Training participants only.

**Primary metric:** Macro F1 is calculated with the fixed 18-class label set for each Validation participant, then averaged **equally across people**. Results from the same Development folds support model selection; they are not an independent final Test estimate.


**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


: 

**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# -------------------- 4. Deep Learning and Compute Device --------------------
from core.computer_device import computer_device
device = computer_device(min_vram_gb=None)

# ------------------------ 5. Study Research Utilities ------------------------
from ninapro_dataset import ensure_numerical_cache
from ninapro_evaluation import select_model, predictions_from_probabilities
from ninapro_models import UNIMODAL_MODELS
from ninapro_experiments import (
    CLASSICAL_MODELS, run_classical_arm, run_unimodal_arm,
    consolidate_unimodal_outputs, evaluate_finalist_seeds,
    consolidate_seed_robustness,
)

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Read Fixed Classes and Paired Development Windows**

Read the window identities, corrected source labels and participant assignments established in Notebook 02. The reserved five Test participants are excluded before any model fitting; all eleven models are assessed on the same five Development validation folds.

The Training and Validation partitions never share a participant, even though adjacent windows may overlap within an individual source recording.


In [3]:
# =========== 02. Read Fixed Classes and Paired Development Windows ===========

NOTEBOOK_NUMBER = 3
index = pd.read_csv(output_dir / "02_window_index.csv")
roles = pd.read_csv(output_dir / "02_participant_split_assignments.csv")
folds = pd.read_csv(output_dir / "02_development_fold_assignments.csv")
classes = np.sort(pd.read_csv(output_dir / "01_label_dictionary.csv").label.to_numpy(int))
role_map = dict(zip(roles.subject_id, roles.role))
fold_map = dict(zip(folds.subject_id, folds.validation_fold))
index["role"] = index.subject_id.map(role_map)
index["validation_fold"] = index.subject_id.map(fold_map)
if index.role.isna().any() or index.loc[index.role == "Development", "validation_fold"].isna().any():
    raise ValueError("Incomplete Notebook 02 person/fold assignments.")

dev = index.loc[index.role == "Development"].copy()
assert not set(dev.subject_id) & set(roles.loc[roles.role == "Test", "subject_id"])
print("Development people:", dev.subject_id.nunique(), "| windows:", len(dev))
print("Independent Development folds:", sorted(dev.validation_fold.unique()))


**03. Recover Aligned Statistical and Temporal Inputs**

Reuse the paired numerical representations established in Notebook 02: **48 statistical sEMG features** and **96 acceleration features** per window, together with the **600-row temporal recordings** required by the CNN, TCN and BiGRU models.

The input routine validates the paired sensor arrays and statistical features. Existing compatible files are reused; unavailable inputs are reconstructed from the local source archives or retrieved when permitted. Console messages summarise data preparation or confirm feature reuse without exposing per-participant cache details.

Both modalities retain the same canonical `window_row` ordering. Feature scaling and neural normalisation are fitted **within Training participants** at the model-fitting procedure; no global cohort-level scaler is fitted here.


In [ ]:
# ============ 03. Recover Aligned Statistical and Temporal Inputs ============

emg_features, acc_features = ensure_numerical_cache(
    root, index, allow_source_download=True, verbose=True,
)
print("Feature array shapes:", emg_features.shape, acc_features.shape)
print("Fixed task classes:", classes.tolist())


**04. Shared Development Protocol**

All eleven models use corrected E1 labels, the same five participant-disjoint Development folds and primary seed `42`. Classical feature scaling and neural channel normalisation are fitted on permitted Training participants. Neural checkpoints use an inner holdout drawn from those Training participants, not the outer Validation group.

Each model produces fold-specific out-of-fold (OOF) probabilities and participant-level metrics. Unimodal sEMG and acceleration models read only their corresponding source channels. On CUDA, the streamed loader supports parallel prefetching; acceleration-only models also prepare normalised fold-specific windows in GPU memory when capacity permits. The same original paired windows, model definitions and training budgets are retained.


In [ ]:
# ====================== 04. Shared Development Protocol ======================

SEED = 42
NEURAL_EPOCHS = 10
NEURAL_PATIENCE = 3
NEURAL_BATCH_SIZE = 96
common_folds = sorted(dev.validation_fold.dropna().unique().astype(int).tolist())

print("Classical models:", ", ".join(CLASSICAL_MODELS))
print("Neural models:", ", ".join(UNIMODAL_MODELS))
print("Grouped Development folds:", common_folds)
print("Training epochs (maximum):", NEURAL_EPOCHS)


**05. B0: Training-Prior Baseline**

This no-signal reference estimates class priors from Training labels and predicts without using either sensor. Its fold-wise macro F1 provides a necessary baseline for interpreting sensor-dependent performance; it does not represent a clinical or movement-aware classifier.


In [ ]:
# ====================== 05. B0: Training-Prior Baseline ======================

# Participant-Grouped Model Fit
display(run_classical_arm(
    "B0_prior", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**06. B1: sEMG Logistic Regression**

Logistic Regression uses the **48 sEMG statistical features** with a Training-fitted standard scaler and balanced class weights. The `lbfgs` solver must reach numerical convergence before its predictions enter the Development comparison. The number of solver iterations is recorded per fold.


In [ ]:
# ===================== 06. B1: sEMG Logistic Regression ======================

#  Participant-Grouped Model Fit
display(run_classical_arm(
    "B1_logreg", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**07. B1: sEMG Random Forest**

The 48-feature sEMG representation is assessed using a class-weighted 120-tree Random Forest. The source windows and independent participant folds are identical to those used by Logistic Regression; no Test data contribute to fitting or comparison.


In [ ]:
# ======================== 07. B1: sEMG Random Forest =========================

# Participant-Grouped Model Fit
display(run_classical_arm(
    "B1_rf", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**08. B2: Acceleration Logistic Regression**

The acceleration-only Logistic Regression receives the **96 statistical acceleration features**. It uses the same regularisation, numerical convergence checks, Training-only standardisation and grouped folds as the sEMG variant.


In [ ]:
# ================= 08. B2: Acceleration Logistic Regression ==================

# Participant-Grouped Model Fit
display(run_classical_arm(
    "B2_logreg", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**09. B2: Acceleration Random Forest**

A 120-tree Random Forest uses only the acceleration feature matrix. Its validation participants, scoring classes and metric aggregation are unchanged, so modality comparisons do not depend on different data partitions.


In [ ]:
# ==================== 09. B2: Acceleration Random Forest =====================

# Participant-Grouped Model Fit 
display(run_classical_arm(
    "B2_rf", dev, emg_features, acc_features, classes,
    output_dir=output_dir, folds=common_folds, seed=SEED,
))


**10. D1: sEMG-Only 1D-CNN**

A compact three-layer 1D-CNN learns local temporal patterns from the **12 sEMG channels**. Its Training-only inner holdout selects the checkpoint; each outer fold reports seven independent Validation participants. Acceleration is excluded from its predictive inputs.


In [ ]:
# ========================= 10. D1: sEMG-Only 1D-CNN ==========================

# Participant-Grouped Model Fit
display(run_unimodal_arm(
    "D1_emg_cnn", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**11. D1: Acceleration-Only 1D-CNN**

The matched 1D-CNN receives **36 acceleration columns** on the released common sample-row grid. The original acceleration acquisition rate was lower than the stored grid rate, so adjacent interpolated rows are not independent high-rate sensor measurements.

For CUDA training, normalised acceleration windows from the permitted Training participants are placed in GPU memory once per fold when sufficient memory is available. Each epoch reuses those windows without rereading the original arrays. The streamed loader remains available when GPU memory is insufficient. This affects computational throughput, not the selected windows or model capacity.


In [ ]:
# ===================== 11. D1: Acceleration-Only 1D-CNN ======================

# Participant-Grouped Model Fit
display(run_unimodal_arm(
    "D1_acc_cnn", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**12. D2: sEMG-Only Temporal CNN**

The residual temporal convolutional network uses dilated convolutional blocks on sEMG. It is compared against the other encoders on the same retained windows, common folds, training budget and fixed label definitions.


In [ ]:
# ====================== 12. D2: sEMG-Only Temporal CNN =======================

# Participant-Grouped Model Fit
display(run_unimodal_arm(
    "D2_emg_tcn", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**13. D2: Acceleration-Only Temporal CNN**

The same residual TCN family processes the source-aligned acceleration sequence. Reporting fold-specific training times makes any modality-dependent runtime costs visible without changing the participant-independent evaluation.


In [ ]:
# ================== 13. D2: Acceleration-Only Temporal CNN ===================

# 1. Participant-Grouped Model Fit
display(run_unimodal_arm(
    "D2_acc_tcn", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**14. D3: sEMG-Only BiGRU**

The bidirectional gated recurrent network models temporal dependencies after pooling the sEMG time axis. Bidirectionality uses the complete offline window; it does not demonstrate causal or real-time operation.


In [ ]:
# ========================== 14. D3: sEMG-Only BiGRU ==========================

# Participant-Grouped Model Fit
display(run_unimodal_arm(
    "D3_emg_bigru", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**15. D3: Acceleration-Only BiGRU**

The acceleration BiGRU completes the six unimodal temporal comparisons. Predictions are evaluated on the same seven unseen Development participants in each fold; its probabilities and participant-level scores enter the common model comparison.


In [ ]:
# ====================== 15. D3: Acceleration-Only BiGRU ======================

# Participant-Grouped Model Fit 
display(run_unimodal_arm(
    "D3_acc_bigru", dev, processed_dir, classes, output_dir=output_dir,
    folds=common_folds, device=device, seed=SEED,
    epochs=NEURAL_EPOCHS, patience=NEURAL_PATIENCE,
    batch_size=NEURAL_BATCH_SIZE, figure_save=save_figure,
))


**16. Consolidate Development Experiments**

Load the model-specific OOF probabilities, predictions and participant-level metrics, retaining the original source-window identities, class order, seed and fold assignment. Five Development validation folds provide matched evidence for comparing all eleven models.

The full-length probability arrays are indexed by the original `window_row`. Development rows contain OOF predictions; reserved Test rows remain unpopulated. The reference unimodal model is selected using the mean fixed-class macro F1 across 35 Development participants, each with equal weight.


In [ ]:
# ================== 16. Consolidate Development Experiments ==================

probabilities, classical_metrics, neural_metrics, neural_history, chosen, summary = (
    consolidate_unimodal_outputs(
        index, dev, classes, output_dir=output_dir, folds=common_folds,
        seed=SEED, neural_settings={
            "epochs": NEURAL_EPOCHS,
            "patience": NEURAL_PATIENCE,
            "batch_size": NEURAL_BATCH_SIZE,
        },
    )
)
print("Development-selected unimodal comparator:", chosen)
display(summary)


**17. Select Neural Finalists for Seed Robustness**

Choose one neural model independently within each modality using Development participant-mean macro F1. Training seeds **43** and **44** repeat each selected model on the same grouped folds; these repeated fits are secondary stability evidence and must not be treated as additional independent participants.


In [ ]:
# ================== 17. Select Neural Robustness Finalists ===================

EXTRA_SEEDS = (43, 44)
emg_finalist, _ = select_model(
    neural_metrics, [name for name in UNIMODAL_MODELS if "_emg_" in name]
)
acc_finalist, _ = select_model(
    neural_metrics, [name for name in UNIMODAL_MODELS if "_acc_" in name]
)
print("sEMG finalist:", emg_finalist)
print("Acceleration finalist:", acc_finalist)


**18. Additional Seeds: Selected sEMG Model**

Repeat the Development-selected sEMG temporal architecture at seeds 43 and 44. Each run uses the same participant separation and Training-only normalisation; each seed/fold combination is recorded separately for stability analysis.


In [ ]:
# ===================== 18. sEMG Finalist Seed Robustness =====================

display(evaluate_finalist_seeds(
    emg_finalist, dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seeds=EXTRA_SEEDS, epochs=NEURAL_EPOCHS,
    patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
))


**19. Additional Seeds: Selected Acceleration Model**

Apply the identical two extra seeds to the Development-selected acceleration model. Keep its fold-wise records separate from the sEMG experiment; aggregate the two finalists only after complete seed and fold coverage is verified.


In [ ]:
# ================= 19. Acceleration Finalist Seed Robustness =================

display(evaluate_finalist_seeds(
    acc_finalist, dev, processed_dir, classes,
    output_dir=output_dir, folds=common_folds, device=device,
    seeds=EXTRA_SEEDS, epochs=NEURAL_EPOCHS,
    patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
))
display(consolidate_seed_robustness(
    (emg_finalist, acc_finalist), output_dir,
    folds=common_folds, seeds=EXTRA_SEEDS,
))


**20. Compare Unimodal Performance and Class-Level Errors**

The model comparison uses mean participant macro F1 across independent Development participants. Learning-loss curves describe optimisation but are not measures of held-out classification performance.

The class-level analysis reuses the saved OOF probabilities from all eleven models, without refitting any classifier. For every original corrected label, it reports pooled window-level precision, recall and F1, class support, and the corresponding metrics averaged equally across participants. Fixed-class participant averages assign zero to an unsupported class, consistently with the primary macro-F1 definition.

The selected model is shown using both a raw window-count confusion matrix and a row-normalised matrix (proportion within each true class), complemented by an 11-model per-class F1 heatmap. The pooled and participant-mean metrics have different aggregation units. All displays concern Development predictions; the reserved Test participants are excluded.


In [ ]:
# ================ 20. Development Comparison and Class Errors ================

# ------------------- 1. Participant-Balanced Model Scores --------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
ranked = summary.iloc[::-1]
axes[0].barh(ranked.model_id, ranked.mean_subject_macro_f1)
axes[0].set(
    xlabel="Mean participant macro F1",
    title="Unimodal Development comparison",
)

# ---------------------- 2. Neural Optimisation History -----------------------
curve = neural_history.groupby("epoch", as_index=False).train_loss.mean()
axes[1].plot(curve.epoch, curve.train_loss, marker="o")
axes[1].set(
    xlabel="Training epoch", ylabel="Mean training loss",
    title="Neural optimisation (descriptive)",
)
for ax in axes:
    ax.grid(alpha=0.2)
save_figure(fig, output_dir / "03_learning_curves.png")
plt.show()

# ----------------------- 3. Per-Class OOF Evidence ---------------------------
from ninapro_evaluation import per_class_oof_report

selected_rows = dev.sort_values("window_row").reset_index(drop=True)
window_positions = selected_rows.window_row.to_numpy(int)
all_class_reports = []
for model_name in summary.model_id:
    model_probabilities = probabilities[model_name][window_positions]
    all_class_reports.append(per_class_oof_report(
        selected_rows, model_probabilities, classes, model_id=model_name,
    ))
class_report = pd.concat(all_class_reports, ignore_index=True)
class_report.to_csv(
    output_dir / "03_all_models_per_class_metrics.csv", index=False,
)
selected_class_report = class_report.loc[
    class_report.model_id == chosen
].copy()
selected_class_report.to_csv(
    output_dir / "03_selected_model_per_class_metrics.csv", index=False,
)

# Equal-person class F1 averages reconstruct the original model-selection
# metric; pooled window F1 is a separate descriptive measurement.
expected_score = float(
    summary.set_index("model_id").loc[chosen, "mean_subject_macro_f1"]
)
reported_score = float(selected_class_report.mean_participant_f1.mean())
if not np.isclose(reported_score, expected_score, atol=1e-9):
    raise ValueError("Per-class and participant-level macro F1 disagree.")
display(selected_class_report[[
    "class_label", "support_windows", "participants_with_support",
    "pooled_precision", "pooled_recall", "pooled_f1",
    "mean_participant_recall", "mean_participant_f1",
]].round(4))

# ----------------- 4. Class-Specific F1 Across Models ------------------------
class_grid = (
    class_report.pivot(
        index="model_id", columns="class_label",
        values="mean_participant_f1",
    )
    .reindex(index=summary.model_id, columns=classes)
)
fig, ax = plt.subplots(figsize=(13, 6), layout="constrained")
image = ax.imshow(class_grid.to_numpy(), aspect="auto", vmin=0, vmax=1)
ax.set(
    xlabel="Corrected E1 class", ylabel="Model",
    title="Development OOF: mean participant F1 by class",
)
ax.set_xticks(np.arange(len(classes)), labels=classes)
ax.set_yticks(np.arange(len(class_grid.index)), labels=class_grid.index)
fig.colorbar(image, ax=ax, label="Mean participant class F1")
save_figure(fig, output_dir / "03_per_class_f1_heatmap.png")
plt.show()

# ------------------- 5. Selected-Model Window Confusion ----------------------
selected_probabilities = probabilities[chosen][window_positions]
actual_labels = selected_rows.label.to_numpy(int)
predicted_labels = predictions_from_probabilities(
    selected_probabilities, classes,
)
cm = confusion_matrix(actual_labels, predicted_labels, labels=classes)
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
image = ax.imshow(cm, aspect="auto")
ax.set(
    xlabel="Predicted corrected class", ylabel="True corrected class",
    title=f"Development OOF confusion (window count): {chosen}",
)
ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
ax.set_yticks(np.arange(len(classes)), labels=classes)
fig.colorbar(image, ax=ax, label="Window count")
save_figure(fig, output_dir / "03_class_errors.png")
plt.show()

# ------------------ 6. Within-Class Normalised Confusion ---------------------
class_support = cm.sum(axis=1, keepdims=True)
normalised_cm = np.divide(
    cm, class_support, out=np.zeros_like(cm, dtype=float),
    where=class_support > 0,
)
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
image = ax.imshow(normalised_cm, aspect="auto", vmin=0, vmax=1)
ax.set(
    xlabel="Predicted corrected class", ylabel="True corrected class",
    title=f"Development OOF confusion (true-class recall): {chosen}",
)
ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
ax.set_yticks(np.arange(len(classes)), labels=classes)
fig.colorbar(image, ax=ax, label="Proportion within true class")
save_figure(fig, output_dir / "03_class_errors_normalised.png")
plt.show()


**21. Scientific Interpretation and Research Outputs**

The Development comparison uses the fixed 18-class task, identical participant-disjoint folds, Training-only normalisation and matched source-window identities. The participant-balanced macro F1 remains the primary selection measure; class-specific recall and F1 show which corrected movements are recognised or confused. Large Rest support can dominate raw counts even when fixed-class macro F1 is modest.

The class report and row-normalised confusion matrix are descriptive evaluations of existing OOF predictions. They do not change classifier fitting, model selection or the independent Test boundary. Notebook 04 reads the matched OOF probabilities and Development metrics to assess whether paired fusion adds value relative to the selected unimodal reference.

The research-output ZIP contains the named Notebook 03 files in `output/`, including the class-specific CSVs and plots.


In [ ]:
# ===================== 21. Research Outputs and Download =====================

# ------------------------- 1. Notebook Output Archive ------------------------
download_outputs_from_colab(
    "multimodal_movement_learning/output",
    f"{NOTEBOOK_NUMBER:02d}",
)
